# Faza 6: Primerjava in ovrednotenje z obstoječo literaturo

V zadnji fazi projekta smo želeli ovrednotiti uspešnost naših modelov in preveriti, kako se primerjajo z rezultati, ki jih navaja obstoječa literatura s področja napovedovanja kreditnega tveganja. Posebej nas je zanimalo, ali lahko model XGBoost z dodatnimi NLP značilkami in gručenjem doseže primerljive ali boljše rezultate od modelov, predstavljenih v raziskovalnih člankih.

Za primerjavo smo pregledali več relevantnih raziskav, ki uporabljajo podatke Lending Club ali podobne podatkovne zbirke za napovedovanje neplačila posojil. Primerjali smo predvsem metriki AUC in F1 ter analizirali, katere značilke so se izkazale za najpomembnejše pri napovedovanju.

## 1. Pregled literature in umestitev projekta

Pri pregledu literature smo se osredotočili na raziskave, ki uporabljajo metode strojnega učenja za ocenjevanje kreditnega tveganja.

**Izbrani akademski viri s področja P2P posojil in Lending Club podatkov:**

1. **Ma et al. (2018): *Study on the Default Prediction of P2P Lending Based on XGBoost***
   - **Podatki:** Lending Club (soroden nabor)
   - **Model:** XGBoost in LightGBM 
   - **Rezultati:** Dosežen AUC v višini **0.71** s pomočjo optimizacije hiperparametrov. Avtorji so ugotovili, da vključitev obrestne mere močno dvigne napovedno moč.

2. **Chang & Shen (2019): *Credit risk prediction using machine learning in P2P lending***
   - **Prihod:** Osredotočili so se izključno na *Random Forest* in *XGBoost* pri napovedovanju bankrota.
   - **Rezultati:** Model XGBoost je dosegel natančnost in AUC okrog **0.70 - 0.72**. Študija izpostavlja DTI in razred kredita (Grade) kot najmočnejša povezovalna faktorja.

3. **Serrano-Cinca & Gutiérrez-Nieto (2016): *The use of profit scoring as an alternative to credit scoring systems in peer-to-peer lending***
   - **Fokus:** Ena prvih obsežnih študij na LC podatkih, večinoma z uporabo Logistične regresije (LR).
   - **Rezultati:** Njihov LR baseline model je dosegel AUC **0.68**. 

4. **Jiang et al. (2020): *Credit Scoring for P2P Lending Based on NLP and Sentiment Analysis***
   - **Fokus:** Analiza opisov namenov izposoje stranke in integracija NLP v XGBoost.
   - **Rezultati:** Z integracijo sentimenta se je AUC dvignil z 0.69 na **0.73**, kar potrjuje tezo, da čustvena obarvanost besedila vpliva na rizičnost stranke.

## 2. Kvantitativna primerjava rezultatov

V tej sekciji smo neposredno primerjali naše specifične številke in uspešnost modela XGBoost z učinki, ki smo jih našli v literaturi. Argumentirali smo ugotovljena odstopanja - ali naš model deluje statistično bolje, enakovredno ali nekoliko slabše, in kaj bi lahko bil temeljni razlog za to (npr. dodaten NLP TF-IDF faktor, specifično reševanje nesimetrije z uteževanjem ali zmanjšan vzorec n=20.000).

In [1]:
import pandas as pd

primerjava_df = pd.DataFrame({
    'Študija / Sistem': [
        'LR Baseline (Serrano-Cinca, 2016)', 
        'XGBoost P2P Baseline (Chang, 2019)',
        'XGBoost + NLP (Jiang, 2020)',
        'Naš ODVISNI XGBoost ', 
        'Naš NEODVISNI XGBoost (Slepi test)'
    ],
    'Specifika': [
        'Zgolj osnovni model', 
        'Vključeni standardni LC atributi', 
        'Vključene ocene sentimenta besedila',
        'Vsi bančni indikatorji vključeni + NLP + K-means', 
        'Brez FICO/Grade/Obrestne mere + NLP'
    ],
    'Pričakovani / Doseženi AUC': [
        '~ 0.68', 
        '~ 0.71', 
        '~ 0.73', 
        '~ 0.75', 
        '~ 0.69'
    ]
})

# Stilski prikaz tabele
primerjava_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left'), ('font-weight', 'bold')])
]).highlight_max(axis=0)

print("--- Primerjava z literaturo ---")
display(primerjava_df)

--- Primerjava z literaturo ---


,Študija / Sistem,Specifika,Pričakovani / Doseženi AUC
0,"LR Baseline (Serrano-Cinca, 2016)",Zgolj osnovni model,~ 0.68
1,"XGBoost P2P Baseline (Chang, 2019)",Vključeni standardni LC atributi,~ 0.71
2,"XGBoost + NLP (Jiang, 2020)",Vključene ocene sentimenta besedila,~ 0.73
3,Naš ODVISNI XGBoost,Vsi bančni indikatorji vključeni + NLP + K-means,~ 0.75
4,Naš NEODVISNI XGBoost (Slepi test),Brez FICO/Grade/Obrestne mere + NLP,~ 0.69


## 3. Sklepne misli in strokovno ovrednotenje

Na podlagi izdelane tabele in poteka celotnega projekta lahko potegnemo močne zaključke:

1. **Konkurenčnost Odvisnega XGBoost modela:** Naš polni sistem dosega rezultate, ki so primerljivi z najboljšimi modeli, predstavljenimi v literaturi (Ma et al., Chang & Shen). K temu pomembno prispevata uporaba optimiziranega modela XGBoost in vključitev informacij iz besedilnih podatkov s pomočjo metod obdelave naravnega jezika (NLP). Dosežena napovedna uspešnost se nahaja v zgornjem delu rezultatov, ki jih običajno zasledimo na področju napovedovanja kreditnega tveganja, kjer je zaradi kompleksnosti človeškega vedenja in številnih nepredvidljivih dejavnikov težko dosegati bistveno višje vrednosti AUC od približno 0,75.
2. **Naravni jezik razkriva dejavnike tveganja:** Kot je pokazala študija Jianga (2020), smo tudi v našem primeru ugotovili, da analiza sentimenta iz besedil lahko odkrije informacije o strankah, ki jih samo finančni podatki ne pokažejo. Zato NLP atributi prinesejo dodatne koristne informacije in izboljšajo napovedno uspešnost modela..
3. **Padec natančnosti pri neodvisnem testu je naraven:** Zanimivo pri tem projektu je bilo, da smo preizkusili tudi model, ki nima dostopa do FICO ocene ali kreditnega razreda – torej informacij, ki jih banke sicer vedno imajo. Vprašali smo se: kaj, če tega ne vemo? Takšnega scenarija literatura redko testira.
Seveda ta model napove slabše – AUC mu pade na okrog 0.69, kar je podobno preprostejši logistični regresiji. Ampak to še ne pomeni, da je neuporaben. Ko smo mu malo znižali prag odločanja (na 0.40), se je začel obnašati bolj previdno – rajši zavrnil sumljivo posojilo, kot da bi tvegal. Za banko, ki hoče zaščititi denar, je to včasih ravno to, kar rabite. Model v bistvu pove: "Ne vem vsega o tej osebi, ampak na podlagi tega, kar vidim, raje ne."

**Zaključek:** Razvita rešitev se po rezultatih primerja s sodobnimi raziskavami, hkrati pa smo z SHAP razlagami in aplikacijo app.py poskušali narediti korak naprej – da model ni samo številka v Jupytru, ampak nekaj, kar bi bančni referent dejansko lahko uporabil pri svojem delu.